# EDA — Olist Customer Reviews

**Data:** 2026-11-06

Exploração inicial do dataset de reviews da Olist. Objetivo é entender a distribuição de scores, volume temporal e características dos textos antes de indexar tudo no ChromaDB.

In [ ]:
import sys
from pathlib import Path

# notebooks/ está um nível abaixo do projeto — adiciona a raiz ao path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid")
print("imports ok")

In [ ]:
from src.ingestion import load_reviews, summarize_reviews

In [ ]:
# carrega o CSV — espera que data/raw/olist_order_reviews_dataset.csv exista
DATA_PATH = Path.cwd().parent / "data" / "raw" / "olist_order_reviews_dataset.csv"

df_raw = pd.read_csv(DATA_PATH)
df_raw.head()

In [ ]:
print("Shape:", df_raw.shape)
print()
print(df_raw.dtypes)

In [ ]:
print("Nulos por coluna:")
print(df_raw.isnull().sum())

**Nota:** Muitos reviews não têm `review_comment_message` preenchido. Esses registros são descartados no `load_reviews()` porque não há texto para indexar. O pipeline usa apenas reviews com texto.

In [ ]:
# distribuição de review_score
score_counts = df_raw["review_score"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
score_counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Distribuição de review_score")
ax.set_xlabel("Score")
ax.set_ylabel("Quantidade")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.xticks(rotation=0)
plt.tight_layout()

img_dir = Path.cwd().parent / "images"
img_dir.mkdir(exist_ok=True)
plt.savefig(img_dir / "eda_score_distribution.png", dpi=150)
plt.show()
print("salvo em images/eda_score_distribution.png")

In [ ]:
# volume por mês-ano
df_raw["review_creation_date"] = pd.to_datetime(df_raw["review_creation_date"])
df_raw["month_year"] = df_raw["review_creation_date"].dt.to_period("M")

monthly = df_raw.groupby("month_year").size().reset_index(name="count")
monthly["month_year_str"] = monthly["month_year"].astype(str)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly["month_year_str"], monthly["count"], marker="o", linewidth=1.5)
ax.set_title("Volume de reviews por mês")
ax.set_xlabel("Mês")
ax.set_ylabel("Quantidade")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# comprimento médio do texto por score
df_com_texto = df_raw.dropna(subset=["review_comment_message"]).copy()
df_com_texto["text_len"] = df_com_texto["review_comment_message"].str.len()

avg_len = df_com_texto.groupby("review_score")["text_len"].mean().round(1)
print("Comprimento médio do texto por score:")
print(avg_len)

In [ ]:
# exemplos por categoria de score
for score in [1, 3, 5]:
    sample = (
        df_com_texto[df_com_texto["review_score"] == score]["review_comment_message"]
        .dropna()
        .sample(n=2, random_state=42)
    )
    print(f"\n--- Score {score} ---")
    for i, text in enumerate(sample, 1):
        print(f"[{i}] {text[:200]}")

## Análises Complementares

In [ ]:
# 1. Distribuição de comprimento de texto (histograma)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_com_texto["text_len"], bins=60, color="steelblue", edgecolor="white", log=True)
ax.axvline(
    df_com_texto["text_len"].median(),
    color="red", linestyle="--",
    label=f"Mediana: {df_com_texto['text_len'].median():.0f} chars",
)
ax.set_title("Distribuição de comprimento dos textos (escala log)")
ax.set_xlabel("Número de caracteres")
ax.set_ylabel("Frequência (log)")
ax.legend()
plt.tight_layout()
plt.show()
print(df_com_texto["text_len"].describe().round(1))

In [ ]:
# 2. Correlação entre score e presença de título (review_comment_title)
df_raw["tem_titulo"] = (
    df_raw["review_comment_title"].notna()
    & (df_raw["review_comment_title"].str.strip() != "")
)
taxa_titulo = df_raw.groupby("review_score")["tem_titulo"].mean().mul(100).round(1)

fig, ax = plt.subplots(figsize=(6, 4))
taxa_titulo.plot(kind="bar", ax=ax, color="coral", edgecolor="white")
ax.set_title("% de reviews com título por score")
ax.set_xlabel("Score")
ax.set_ylabel("% com título")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print("% reviews com título por score:")
print(taxa_titulo)

In [ ]:
# 3. Reviews duplicados — mesmo order_id com scores diferentes
dup = df_raw.groupby("order_id").filter(lambda g: g["review_score"].nunique() > 1)
print(f"Orders com scores conflitantes: {dup['order_id'].nunique()}")
print(f"Reviews envolvidos:              {len(dup)}")
if len(dup) > 0:
    print("\nExemplo:")
    print(dup[["order_id", "review_score", "review_comment_message"]].head(4).to_string())

In [ ]:
# 4. Sazonalidade — Black Friday aparece em novembro?
df_raw["month"] = df_raw["review_creation_date"].dt.month
vol_mensal = df_raw.groupby("month").size()

fig, ax = plt.subplots(figsize=(8, 4))
vol_mensal.plot(kind="bar", ax=ax, color="mediumseagreen", edgecolor="white")
ax.set_title("Volume de reviews por mês (todos os anos)")
ax.set_xlabel("Mês")
ax.set_ylabel("Quantidade")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
meses = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
ax.set_xticklabels(meses, rotation=0)
plt.tight_layout()
plt.show()

media = vol_mensal.mean()
nov = vol_mensal.get(11, 0)
print(f"Novembro: {nov:,} reviews | Média mensal: {media:,.0f} | Ratio: {nov/media:.2f}x")

In [ ]:
# 5. Encoding — acentos estão OK?
amostras = df_com_texto["review_comment_message"].dropna().head(50).tolist()
chars_acentuados = "áéíóúãõâêôàüçÁÉÍÓÚÃÕÂÊÔÀÜÇ"
com_acento = [t for t in amostras if any(c in t for c in chars_acentuados)]
print(f"Reviews com acentos (amostra 50): {len(com_acento)}/50")
print("\nExemplo com acento:")
if com_acento:
    print(f"  {com_acento[0][:150]}")

In [ ]:
# 6. load_reviews() vs df_raw — quantos ficam após filtro de texto vazio
reviews = load_reviews()
total_raw = len(df_raw)
total_com_texto = len(reviews)
descartados = total_raw - total_com_texto

print(f"df_raw total:             {total_raw:>7,}")
print(f"load_reviews() retorna:   {total_com_texto:>7,}  ({total_com_texto/total_raw*100:.1f}%)")
print(f"Descartados (sem texto):  {descartados:>7,}  ({descartados/total_raw*100:.1f}%)")
print(f"\n→ 6 em cada 10 reviews não têm comentário escrito — só score.")